# 02. Data Preparation and Splitting

This notebook executes the data cleaning, splitting, and filtering pipeline:
1. **Cleaning**: Normalizes whitespace and drops duplicate translation pairs (removes 51 pairs).
2. **Splitting**: Splits the 669,094 pairs into Train (80%), Validation (10%), and Test (10%).
3. **Filtering**: Applies `MAX_LEN = 70` (including `<SOS>` and `<EOS>`) to produce final filtered datasets.
4. **Export**: Saves processed CSV files to disk / Google Drive under `PROCESSED_DIR`.


In [ ]:
# Environment & Path Setup
# If running on Google Colab, uncomment the lines below:
# from google.colab import drive
# drive.mount('/content/drive')
# %cd /content/english-amharic-nmt

import os
import sys
from pathlib import Path

# Add project root to sys.path
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.utils.paths import get_data_paths
from src.utils.seed import set_seed

# Configure data directory (can be overridden by DATA_ROOT environment variable)
# On Colab: DATA_ROOT = "/content/drive/MyDrive/english-amharic-nmt-data"
DATA_ROOT = os.getenv("DATA_ROOT", str(PROJECT_ROOT / "data"))
paths = get_data_paths(DATA_ROOT)
paths.ensure_directories()
set_seed(42)

print("Project root:", PROJECT_ROOT)
print("Data directory:", paths.data_root)


## 1. Load Raw Dataset

In [ ]:
from src.data.dataset import load_raw_dataset_from_hf

raw_df = load_raw_dataset_from_hf(cache_dir=paths.raw_dir)
print(f"Raw dataset shape: {raw_df.shape}")


## 2. Clean Corpus & Remove Duplicate Pairs
Using `clean_parallel_corpus` from `src.data.preprocessing`:
- Drops any NA values.
- Normalizes whitespace with regex `\s+ -> ' '`.
- Removes exact duplicate sentence pairs.


In [ ]:
from src.data.preprocessing import clean_parallel_corpus

before = len(raw_df)
clean_df = clean_parallel_corpus(raw_df)
after = len(clean_df)

print(f"Before cleaning: {before:,}")
print(f"After removing duplicate pairs: {after:,}")
print(f"Removed duplicates: {before - after:,}")
assert after == 669094, f"Expected 669,094 pairs, got {after}"


## 3. Train / Validation / Test Split (80 / 10 / 10)
We split with `random_state=42` and verify that splits have zero mutual overlap.

In [ ]:
from src.data.preprocessing import split_data

train_df, val_df, test_df = split_data(clean_df, test_size=0.20, val_ratio_of_temp=0.50, random_state=42)

print(f"Train set:      {len(train_df):,d} ({len(train_df)/len(clean_df)*100:.1f}%)")
print(f"Validation set: {len(val_df):,d} ({len(val_df)/len(clean_df)*100:.1f}%)")
print(f"Test set:       {len(test_df):,d} ({len(test_df)/len(clean_df)*100:.1f}%)")
print(f"Total:          {len(train_df) + len(val_df) + len(test_df):,d}")

# Verify no overlap between splits
train_pairs = set(zip(train_df["eng"], train_df["amh"]))
val_pairs = set(zip(val_df["eng"], val_df["amh"]))
test_pairs = set(zip(test_df["eng"], test_df["amh"]))

print(f"Train ∩ Validation overlap: {len(train_pairs & val_pairs)}")
print(f"Train ∩ Test overlap:       {len(train_pairs & test_pairs)}")
print(f"Validation ∩ Test overlap:  {len(val_pairs & test_pairs)}")


## 4. Save Unfiltered Datasets
We save the unfiltered 80/10/10 splits (`train.csv`, `validation.csv`, `test.csv`).

In [ ]:
train_df.to_csv(paths.train_path, index=False)
val_df.to_csv(paths.val_path, index=False)
test_df.to_csv(paths.test_path, index=False)

print("Unfiltered datasets saved:")
for p in [paths.train_path, paths.val_path, paths.test_path]:
    size_mb = p.stat().st_size / (1024 * 1024)
    print(f" - {p.name:<20} {size_mb:6.2f} MB")


## 5. Sequence Length Filtering (MAX_LEN = 70)
Every sequence will later be wrapped with `<SOS>` and `<EOS>`. Therefore:
$$\text{token\_length} + 2 \le 70$$
Sentence pairs exceeding 70 tokens in either English or Amharic are removed from training, validation, and test sets.


In [ ]:
from src.data.preprocessing import filter_by_max_length

MAX_LEN = 70

train_filtered = filter_by_max_length(train_df, max_len=MAX_LEN)
val_filtered = filter_by_max_length(val_df, max_len=MAX_LEN)
test_filtered = filter_by_max_length(test_df, max_len=MAX_LEN)

print("Filtered split sizes:")
print(f"Train:      {len(train_filtered):,d} (Kept: {len(train_filtered)/len(train_df)*100:.2f}%)")
print(f"Validation: {len(val_filtered):,d} (Kept: {len(val_filtered)/len(val_df)*100:.2f}%)")
print(f"Test:       {len(test_filtered):,d} (Kept: {len(test_filtered)/len(test_df)*100:.2f}%)")
print(f"Total:      {len(train_filtered) + len(val_filtered) + len(test_filtered):,d}")


## 6. Save Filtered Datasets to Processed Directory

In [ ]:
train_filtered.to_csv(paths.train_filtered_path, index=False)
val_filtered.to_csv(paths.val_filtered_path, index=False)
test_filtered.to_csv(paths.test_filtered_path, index=False)

print("Filtered datasets saved:")
for p in [paths.train_filtered_path, paths.val_filtered_path, paths.test_filtered_path]:
    size_mb = p.stat().st_size / (1024 * 1024)
    print(f" - {p.name:<25} {size_mb:6.2f} MB")
